# 🧪 معمل التجارب: بناء مسار RAG باستخدام LCEL ومشغل الربط (|)
كود بسيط ومباشر يطبق المحاضرة 28 بدون دوال مغلقة وبشفافية كاملة.

In [8]:
# استدعاء المكتبات ومسار البيانات المركزي
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

data_dir = Path("../../data") if Path("../../data").exists() else Path("./data")
docs_dir = str(data_dir / "chroma_sample_data")
db_dir = str(data_dir / "databases" / "chroma_vector_store")

# مكتبات LCEL الأساسية
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("المكتبات جاهزة!")

المكتبات جاهزة!


In [ ]:
# 1. قراءة النصوص وتقطيعها وتجهيز الـ Retriever
loader = DirectoryLoader(docs_dir, glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
raw_documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
documents = splitter.split_documents(raw_documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# استدعاء الفهرس من مسار data/databases
vector_store = Chroma(
    persist_directory=db_dir,
    embedding_function=embeddings,
    collection_name="rag_collection"
)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(f"المسترجع جاهز للبحث بين {vector_store._collection.count()} قطعة!")

In [ ]:
# 2. دالة بسيطة لدمج نصوص المستندات المسترجعة (بديل دالة Stuff Chain)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# تجربة سريعة للدالة على مخرجات المسترجع
test_docs = retriever.invoke("What is deep learning?")
print("مثال للسياق المدمج:")
print(format_docs(test_docs)[:140] + "...")

In [ ]:
# 3. إعداد قالب الـ Prompt ونموذج Groq
template = """You are a helpful AI assistant. Use the following context to answer the question.
If you don't know the answer, say you don't know.
Keep the answer concise.

Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)
print("الـ Prompt والموديل جاهزان!")

In [ ]:
# 4. بناء السلسلة الكاملة بمشغل الربط (| Pipe Operator)
# - retriever | format_docs: يجلب المستندات ويدمج نصوصها ليضعها في {context}
# - RunnablePassthrough(): يمرر السؤال النصي زي ما هو ليوضع في {question}
# - prompt: يدمج الاثنين في رسالة واحدة للموديل
# - llm: يولد الرد
# - StrOutputParser(): يرجع النص الصافي فقط
rag_chain_lcel = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("تم بناء سلسلة الـ LCEL بنجاح!")

In [ ]:
# 5. تجربة السؤال المباشر كنص (String)
query = "What is deep learning and how does it differ from traditional machine learning?"
answer = rag_chain_lcel.invoke(query)

print(f"❓ السؤال: {query}\n")
print(f"💬 الإجابة المباشرة:\n{answer}")

In [ ]:
# 6. تجربة البث الحرفي الفوري كلمة بكلمة (Streaming)
# دي ميزة جبارة في LCEL بتخلي الإجابة تظهر لحظة بلحظة زي ChatGPT!
stream_query = "What are CNNs best used for?"
print(f"⚡ بدء البث المباشر لسؤال: '{stream_query}'\n")

for chunk in rag_chain_lcel.stream(stream_query):
    print(chunk, end="", flush=True)
print()